# using "qwen2.5:7b-instruct" model

In [7]:
import json
import requests
from tqdm import tqdm
import os
import re
import time

# configuration 

In [8]:
# =========================
# CONFIG
# =========================

MODEL_NAME = "qwen2.5:7b-instruct"

# INPUT_PATH = r"/content/drive/MyDrive/Minor Project/CheckThat-2025/1-my dataset/1-train-set-claims.json"
# OUTPUT_PATH = r"/content/drive/MyDrive/Minor Project/CheckThat-2025/1-my dataset/1-decomposed-claims.json"

INPUT_PATH = r"E:\Projects\Numerical Fact Checking\CheckThat 2025\task3\data\1-my dataset\2-val-set-claims.json"
OUTPUT_PATH = r"E:\Projects\Numerical Fact Checking\CheckThat 2025\task3\data\1-my dataset\2-decomposed-claims.json"

OLLAMA_URL = "http://localhost:11434/api/generate"

MAX_RETRIES = 3
TEMPERATURE = 0.1  # low for structured output

# Prompt Template

In [9]:
def build_prompt(claim):
    prompt = f"""
You are an expert in numerical fact-checking.

Decompose the following claim into EXACTLY 5 independent yes/no questions.

STRICT RULES:
1. Do NOT change any numerical values.
2. Do NOT modify temporal expressions (years, dates, time phrases).
3. Each question must test one atomic fact.
4. Return ONLY valid JSON.
5. Output format:

{{
  "questions": [
    "Question 1?",
    "Question 2?",
    "Question 3?",
    "Question 4?",
    "Question 5?"
  ]
}}

CLAIM:
"{claim}"
"""
    return prompt

# Ollama API Call Function

In [10]:
def call_ollama(prompt):
    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "temperature": TEMPERATURE,
        "stream": False
    }

    response = requests.post(OLLAMA_URL, json=payload)

    if response.status_code != 200:
        raise Exception(f"Ollama Error: {response.text}")

    return response.json()["response"]

# Safe JSON Extraction

In [11]:
def extract_json(text):
    try:
        return json.loads(text)
    except:
        # Try extracting JSON block
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            return json.loads(match.group())
        else:
            raise ValueError("No valid JSON found")

# Validation Function

This ensures:

1. Exactly 5 questions
2. Numbers preserved

In [12]:
def validate_output(original_claim, questions):
    if len(questions) != 5:
        return False
    
    # Extract numbers from original claim
    original_numbers = re.findall(r'\d[\d,]*', original_claim)
    
    # Combine all questions text
    combined_questions = " ".join(questions)
    question_numbers = re.findall(r'\d[\d,]*', combined_questions)
    
    # Ensure numbers unchanged
    for num in original_numbers:
        if num not in question_numbers:
            return False
    
    return True

# **Training Set**

# Main Processing Loop

In [14]:
import os
import json
import time
from tqdm import tqdm

# ==========================
# FILE PATHS (Windows Safe)
# ==========================

INPUT_PATH = r"E:\Projects\Numerical Fact Checking\CheckThat 2025\task3\data\1-my dataset\1-train-set-claims.json"
OUTPUT_PATH = r"E:\Projects\Numerical Fact Checking\CheckThat 2025\task3\data\1-my dataset\1-train-decomposed-claims.json"

MAX_RETRIES = 3


# ==========================
# LOAD INPUT FILE
# ==========================

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Total input claims: {len(data)}")


# ==========================
# LOAD EXISTING OUTPUT
# ==========================

if os.path.exists(OUTPUT_PATH):
    try:
        with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
            results = json.load(f)

        if not isinstance(results, list):
            print("⚠ Output file corrupted. Resetting.")
            results = []

    except json.JSONDecodeError:
        print("⚠ Output file invalid JSON. Resetting.")
        results = []
else:
    results = []

# Create processed ID set for fast lookup
processed_ids = {item["original_id"] for item in results}

print(f"Already processed: {len(processed_ids)}")


# ==========================
# MAIN LOOP
# ==========================

for item in tqdm(data):

    claim_id = item["id"]
    claim_text = item["claim"]

    # 🔥 Skip if already decomposed
    if claim_id in processed_ids:
        continue

    success = False

    for attempt in range(MAX_RETRIES):
        try:
            prompt = build_prompt(claim_text)
            raw_output = call_ollama(prompt)
            parsed = extract_json(raw_output)

            if "questions" not in parsed:
                raise ValueError("Missing 'questions' key")

            questions = parsed["questions"]

            if not isinstance(questions, list) or len(questions) == 0:
                raise ValueError("Questions not valid list")

            if validate_output(claim_text, questions):

                result_entry = {
                    "original_id": claim_id,
                    "original_claim": claim_text,
                    "decomposed_questions": questions
                }

                results.append(result_entry)
                processed_ids.add(claim_id)

                # 💾 Save immediately (crash-safe)
                with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
                    json.dump(results, f, indent=2, ensure_ascii=False)

                success = True
                break

            else:
                print(f"Validation failed for ID {claim_id}, retrying...")
                print(time.strftime(" %H:%M:%S", time.localtime()))
                time.sleep(1)

        except Exception as e:
            print(f"Error on ID {claim_id}: {e}")
            time.sleep(1)

    if not success:
        # print(f"❌ Failed permanently on ID {claim_id}")
        continue


print("✅ Decomposition complete.")



Total input claims: 9935
Already processed: 9467


  0%|          | 10/9935 [00:38<10:40:41,  3.87s/it]


KeyboardInterrupt: 

# re run cell

In [ ]:
# -------------------------
# CONTROLLED RE-RUN CELL
# -------------------------
# Start at a specific 1-based claim index, with safe id checkpointing.

START_INDEX = 11 

start_idx = max(0, START_INDEX - 1)

print(f"Restarting from 1-based claim index: {START_INDEX} (0-based {start_idx})")

for idx in range(start_idx, len(data)):
    item = data[idx]
    claim_id = item["id"]
    claim_text = item["claim"]

    if claim_id in processed_ids:
        # Already done from previous run
        continue

    success = False
    for attempt in range(MAX_RETRIES):
        try:
            prompt = build_prompt(claim_text)
            raw_output = call_ollama(prompt)
            parsed = extract_json(raw_output)

            if "questions" not in parsed:
                raise ValueError("Missing 'questions' key")

            questions = parsed["questions"]
            if not isinstance(questions, list) or len(questions) == 0:
                raise ValueError("Questions not valid list")

            if validate_output(claim_text, questions):
                result_entry = {
                    "original_id": claim_id,
                    "original_claim": claim_text,
                    "decomposed_questions": questions
                }
                results.append(result_entry)
                processed_ids.add(claim_id)

                with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
                    json.dump(results, f, indent=2, ensure_ascii=False)

                print(f"Saved claim {claim_id} at data index {idx}")
                success = True
                break
            else:
                print(f"Validation failed for ID {claim_id}, attempt {attempt + 1}/{MAX_RETRIES}")
                time.sleep(1)

        except Exception as e:
            print(f"Error on ID {claim_id} (attempt {attempt + 1}/{MAX_RETRIES}): {e}")
            time.sleep(1)

    if not success:
        print(f"Skipping claim {claim_id} after {MAX_RETRIES} attempts")

print("✅ Controlled re-run complete.")

In [ ]:
# 1303 thayu che lala
# 1877 thai gayu


# Test Set Again:
# 3023 hatu before start
# 3066 thayu pachi


# Save Output File

In [16]:
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

print("✅ Decomposition complete and saved.")

✅ Decomposition complete and saved.


# checking how many are decomposed after saving

In [17]:
with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    results = json.load(f)
print(f"Total decomposed claims: {len(results)}")

Total decomposed claims: 9467


# Checks which are decomposed and which are not decoposed 


In [23]:
train_set=r"E:\Projects\Numerical Fact Checking\CheckThat 2025\task3\data\1-my dataset\1-train-set-claims.json"
train_decomposed=r"E:\Projects\Numerical Fact Checking\CheckThat 2025\task3\data\1-my dataset\1-train-decomposed-claims.json"
val_set=r"E:\Projects\Numerical Fact Checking\CheckThat 2025\task3\data\1-my dataset\2-val-set-claims.json"
val_decomposed=r"E:\Projects\Numerical Fact Checking\CheckThat 2025\task3\data\1-my dataset\2-decomposed-claims.json"
test_set=r"E:\Projects\Numerical Fact Checking\CheckThat 2025\task3\data\1-my dataset\3-test-set-claims.json"
test_decomposed=r"E:\Projects\Numerical Fact Checking\CheckThat 2025\task3\data\1-my dataset\3-decomposed-claims.json"

In [24]:
import json

pairs = [
	("train", train_set, train_decomposed),
	("val", val_set, val_decomposed),
	("test", test_set, test_decomposed),
]

for name, src_path, dec_path in pairs:
	with open(src_path, "r", encoding="utf-8") as f:
		src_data = json.load(f)
	with open(dec_path, "r", encoding="utf-8") as f:
		dec_data = json.load(f)

	dec_map = {}
	for item in dec_data:
		# some files use original_id
		key = item.get("original_id", item.get("id"))
		if key is None:
			continue
		dec_map[key] = item

	total_claims = len(src_data)
	valid_decomposed = 0
	not_decomposed_ids = []

	for item in src_data:
		claim_id = item.get("id")
		if claim_id is None:
			continue

		dec_item = dec_map.get(claim_id)
		if dec_item is None:
			not_decomposed_ids.append(claim_id)
			continue

		questions = dec_item.get("decomposed_questions")
		if questions is None:
			questions = dec_item.get("questions", [])
		if not isinstance(questions, list) or len(questions) < 5:
			not_decomposed_ids.append(claim_id)
		else:
			valid_decomposed += 1

	print(f"{name.upper()}: total claims = {total_claims}")
	print(f"  valid decomposed (>=5 questions) = {valid_decomposed}")
	print(f"  not decomposed or incomplete (<5) = {len(not_decomposed_ids)}")
	print()

TRAIN: total claims = 9935
  valid decomposed (>=5 questions) = 9467
  not decomposed or incomplete (<5) = 468

VAL: total claims = 3084
  valid decomposed (>=5 questions) = 2826
  not decomposed or incomplete (<5) = 258

TEST: total claims = 3656
  valid decomposed (>=5 questions) = 3124
  not decomposed or incomplete (<5) = 532



# checks whether all are in continous order or not

In [25]:
# check matching ids and claims between source and decomposed files
pairs = [
	("TRAIN", train_set, train_decomposed),
	("VAL", val_set, val_decomposed),
	("TEST", test_set, test_decomposed),
]

for name, src_path, dec_path in pairs:
	with open(src_path, "r", encoding="utf-8") as f:
		src_data = json.load(f)
	with open(dec_path, "r", encoding="utf-8") as f:
		dec_data = json.load(f)

	src_map = {item["id"]: item for item in src_data if "id" in item}
	dec_map = {
		(item.get("original_id") if item.get("original_id") is not None else item.get("id")): item
		for item in dec_data
		if item.get("original_id") is not None or item.get("id") is not None
	}

	src_ids = set(src_map.keys())
	dec_ids = set(dec_map.keys())

	missing_in_dec = sorted(src_ids - dec_ids)
	extra_in_dec = sorted(dec_ids - src_ids)
	common_ids = sorted(src_ids & dec_ids)

	mismatched_claims = []
	for cid in common_ids:
		src_claim = str(src_map[cid].get("claim", "")).strip().lower()
		dec_claim = str(dec_map[cid].get("original_claim", dec_map[cid].get("claim", ""))).strip().lower()
		if src_claim != dec_claim:
			mismatched_claims.append((cid, src_map[cid].get("claim"), dec_map[cid].get("original_claim", dec_map[cid].get("claim"))))

	print(f"--- {name} ---")
	print("source total:", len(src_ids))
	print("decomposed total:", len(dec_ids))
	print("missing ids in decomposed:", len(missing_in_dec))
	print("extra ids in decomposed:", len(extra_in_dec))
	print("common ids:", len(common_ids))
	print("claim text mismatches on common ids:", len(mismatched_claims))
	if missing_in_dec:
		print("  sample missing:", missing_in_dec[:20])
	if extra_in_dec:
		print("  sample extra:", extra_in_dec[:20])
	if mismatched_claims:
		print("  sample mismatches:", mismatched_claims[:5])
	print()

--- TRAIN ---
source total: 9935
decomposed total: 9467
missing ids in decomposed: 468
extra ids in decomposed: 0
common ids: 9467
claim text mismatches on common ids: 0
  sample missing: [11, 78, 81, 88, 95, 196, 204, 223, 229, 301, 302, 312, 329, 332, 353, 383, 398, 421, 435, 480]

--- VAL ---
source total: 3084
decomposed total: 2826
missing ids in decomposed: 258
extra ids in decomposed: 0
common ids: 2826
claim text mismatches on common ids: 0
  sample missing: [9, 28, 62, 120, 123, 139, 142, 159, 173, 176, 199, 233, 240, 262, 275, 279, 312, 329, 349, 354]

--- TEST ---
source total: 3656
decomposed total: 3124
missing ids in decomposed: 532
extra ids in decomposed: 0
common ids: 3124
claim text mismatches on common ids: 0
  sample missing: [76, 127, 135, 149, 150, 160, 174, 191, 197, 236, 240, 268, 315, 321, 367, 425, 426, 440, 444, 452]



# removing all the wrongly decomposed things

In [5]:
# remove mismatched claim text entries from val_decomposed
with open(val_set, "r", encoding="utf-8") as f:
    val_src = json.load(f)

with open(val_decomposed, "r", encoding="utf-8") as f:
    val_dec = json.load(f)

src_claim_by_id = {
    item["id"]: str(item["claim"]).strip().lower()
    for item in val_src
    if "id" in item and "claim" in item
}

cleaned = []
removed_ids = []

for entry in val_dec:
    oid = entry.get("original_id", entry.get("id"))
    if oid is None:
        cleaned.append(entry)
        continue

    if oid in src_claim_by_id:
        src_claim = src_claim_by_id[oid]
        dec_claim = str(entry.get("original_claim", entry.get("claim", ""))).strip().lower()

        if src_claim != dec_claim:
            removed_ids.append(oid)
            continue

    cleaned.append(entry)

print(f"Removed mismatched entries: {len(removed_ids)}")
print(f"Remaining entries: {len(cleaned)}")

with open(val_decomposed, "w", encoding="utf-8") as f:
    json.dump(cleaned, f, indent=2, ensure_ascii=False)

Removed mismatched entries: 0
Remaining entries: 2902


# making new json file for train set


In [6]:
# Load train set and decomposed data
with open(train_set, "r", encoding="utf-8") as f:
    src_data = json.load(f)
with open(train_decomposed, "r", encoding="utf-8") as f:
    dec_data = json.load(f)

# Create map of decomposed claims
dec_map = {}
for item in dec_data:
    key = item.get("original_id", item.get("id"))
    if key is None:
        continue
    questions = item.get("decomposed_questions", item.get("questions", []))
    if isinstance(questions, list) and len(questions) >= 5:
        dec_map[key] = True

# Find not decomposed ids
not_decomposed_ids = []
for item in src_data:
    claim_id = item.get("id")
    if claim_id and claim_id not in dec_map:
        not_decomposed_ids.append(claim_id)

print("Not decomposed claim IDs in training set:")
print(not_decomposed_ids)

Not decomposed claim IDs in training set:
[11, 78, 81, 88, 95, 196, 204, 223, 229, 301, 302, 312, 329, 332, 353, 383, 398, 421, 435, 480, 547, 570, 594, 615, 712, 726, 747, 756, 765, 770, 790, 818, 888, 890, 897, 926, 928, 947, 965, 983, 986, 1035, 1037, 1041, 1045, 1185, 1214, 1230, 1287, 1290, 1295, 1302, 1313, 1318, 1319, 1356, 1385, 1426, 1440, 1468, 1495, 1510, 1591, 1635, 1641, 1645, 1698, 1703, 1708, 1726, 1750, 1793, 1794, 1810, 1821, 1837, 1843, 1847, 1854, 1855, 1897, 1911, 1921, 1925, 1980, 2007, 2009, 2018, 2041, 2066, 2073, 2083, 2089, 2091, 2097, 2099, 2104, 2112, 2113, 2121, 2149, 2169, 2185, 2229, 2231, 2248, 2309, 2355, 2368, 2372, 2383, 2391, 2491, 2510, 2565, 2625, 2630, 2648, 2653, 2664, 2668, 2675, 2681, 2691, 2699, 2765, 2816, 2828, 2855, 2859, 2884, 2885, 2891, 2908, 2926, 2928, 2943, 2967, 2999, 3040, 3058, 3068, 3084, 3126, 3135, 3138, 3158, 3171, 3175, 3193, 3219, 3229, 3231, 3236, 3274, 3298, 3356, 3372, 3400, 3404, 3409, 3419, 3500, 3518, 3549, 3554, 3559, 3

In [16]:
# Load train set data (assuming src_data is already loaded from cell 31)
# not_decomposed_ids is from cell 31 for train set

new_output_path = r"E:\Projects\Numerical Fact Checking\CheckThat 2025\task3\data\1-my dataset\1-new-decomposed.json"

results_new = []

MAX_RETRIES=3

for claim_id in tqdm(not_decomposed_ids, desc="Processing claims"):
    # Find the claim in src_data
    claim_text = None
    for item in src_data:
        if item["id"] == claim_id:
            claim_text = item["claim"]
            break
    
    if claim_text is None:
        continue
    
    success = False
    for attempt in range(MAX_RETRIES):
        try:
            prompt = build_prompt(claim_text)
            raw_output = call_ollama(prompt)
            parsed = extract_json(raw_output)
            
            if "questions" not in parsed:
                raise ValueError("Missing 'questions' key")
            
            questions = parsed["questions"]
            if not isinstance(questions, list) or len(questions) == 0:
                raise ValueError("Questions not valid list")
            
            if validate_output(claim_text, questions):
                result_entry = {
                    "original_id": claim_id,
                    "original_claim": claim_text,
                    "decomposed_questions": questions
                }
                results_new.append(result_entry)
                success = True
                break
            else:
                time.sleep(1)
        except Exception as e:
            time.sleep(1)
    
    if not success:
        continue

# Save to new file
with open(new_output_path, "w", encoding="utf-8") as f:
    json.dump(results_new, f, indent=2, ensure_ascii=False)

print(f"New decomposed claims saved to {new_output_path}. Total: {len(results_new)}")

Processing claims:   0%|          | 0/494 [00:00<?, ?it/s]

Processing claims: 100%|██████████| 494/494 [5:55:08<00:00, 43.14s/it]  

New decomposed claims saved to E:\Projects\Numerical Fact Checking\CheckThat 2025\task3\data\1-my dataset\1-new-decomposed.json. Total: 26


In [20]:
# Load the old decomposed file
with open(train_decomposed, "r", encoding="utf-8") as f:
    old_results = json.load(f)

# Append the new results
old_results.extend(results_new)

# Save back to the old file
with open(train_decomposed, "w", encoding="utf-8") as f:
    json.dump(old_results, f, indent=2, ensure_ascii=False)

print(f"Combined file saved to {train_decomposed}. Total entries: {len(old_results)}")

Combined file saved to E:\Projects\Numerical Fact Checking\CheckThat 2025\task3\data\1-my dataset\1-train-decomposed-claims.json. Total entries: 9467
